# NALAPRO Project — Part 5 (Bonus)
## Llama-3 — QLoRA Fine-tuning for 20-Newsgroups Classification

---

## Tools used
| Tool | Purpose |
|------|---------|
| `transformers` | Load Llama-3 model and tokenizer |
| `peft` | LoRA adapter configuration (`LoraConfig`, `get_peft_model`) |
| `bitsandbytes` | 4-bit NF4 quantization (`BitsAndBytesConfig`) |
| `trl` | Supervised fine-tuning loop (`SFTTrainer`) |
| `datasets` | Dataset formatting for SFT |
| `scikit-learn` | Evaluation metrics, confusion matrix |
| `matplotlib` | Visualisations |
| `pandas` | Results tables |
| `Claude (Anthropic)` | Scaffolding and code assistance |

**Reference:** *Hands-On Large Language Models* — https://github.com/HandsOnLLM/Hands-On-Large-Language-Models
**QLoRA tutorial:** DataCamp — https://www.datacamp.com/tutorial/fine-tuning-llama-3-1
**QLoRA paper:** Dettmers et al. (2023) — "QLoRA: Efficient Finetuning of Quantized LLMs"
**In-class notebook:** `AdvancedTechniques_LangChain.ipynb`

---

### What is QLoRA?

**QLoRA** (Quantized Low-Rank Adaptation) combines two efficiency techniques:

| Technique | What it does | Benefit |
|-----------|-------------|---------|
| **4-bit NF4 quantization** | Stores base model weights in 4 bits instead of 16 | ~4× smaller memory footprint |
| **LoRA** | Adds small trainable rank-decomposition matrices; base weights stay frozen | Only ~0.1–1% of parameters are actually trained |

Together they make fine-tuning a **7–8B parameter LLM feasible on a single T4 GPU (16 GB VRAM)**.

### Why fine-tune vs. prompt?

| Approach | Training examples | Gradient updates | Part |
|----------|-----------------|-----------------|------|
| Zero-shot (Part 4) | 0 | None | 4 |
| Few-shot k=1 (Part 4) | 20 | None | 4 |
| **QLoRA fine-tune (this part)** | ~11 k | LoRA adapters only | **5** |

**Hypothesis:** Fine-tuning on the full labelled training set should substantially
outperform prompting alone — the model adapts its generation distribution to the task.

---

In [ ]:
# IMPORTANT: Run on Google Colab with a GPU runtime.
# Runtime → Change runtime type → T4 GPU (or A100/L4 for faster training).
# bitsandbytes 4-bit quantisation requires CUDA and will NOT work on CPU or Apple Silicon.
# NOTE: we do NOT upgrade torch here — Colab ships a torch+torchvision pair that must stay in sync.
%pip install -q -U transformers peft bitsandbytes trl datasets accelerate scikit-learn matplotlib pandas tqdm
%pip install -q -U huggingface_hub

---
## Experiment tracking (in-notebook)

Part 5 uses no validation split during training (instruction fine-tuning converges quickly).
The following are tracked:

**Training diagnostics**
- Loss per step from `trainer.state.log_history`
- LoRA parameter count vs. total frozen base-model parameters

**Inference results (same eval protocol as Part 4)**
- `test_acc` — fraction of 400 eval documents classified correctly
- `test_macro_f1` — macro-F1 across all 20 classes
- `unparseable_pct` — fraction of outputs that the label parser could not match

**Cross-part comparison**
- Direct comparison to Part 4 zero-shot and few-shot results (loaded from `part4_results_all.csv`)
- And to Parts 1–3 via their saved CSVs

In [ ]:
import os, random, re, math
import numpy as np
import torch
from collections import defaultdict
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Hardware check ────────────────────────────────────────────────────────────
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected!\n"
        "QLoRA requires CUDA. Go to Runtime → Change runtime type → GPU (T4 or better)."
    )

DEVICE = torch.device("cuda")
print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Google Drive setup (Google Colab only) ───────────────────────────────────
import os

_IN_COLAB = False
NALAPRO_DIR = '/content'
CKPT_BASE   = '/content'

try:
    from google.colab import drive
    try:
        drive.mount('/content/drive')
        NALAPRO_DIR = '/content/drive/MyDrive/NALAPRO'
        os.makedirs(NALAPRO_DIR, exist_ok=True)
        os.chdir(NALAPRO_DIR)
        print(f'Google Drive mounted.')
        print(f'Working directory (CSVs/plots) → {NALAPRO_DIR}')
    except Exception as e:
        print(f'Drive mount failed ({e}). Saving to local /content instead.')
        print('Files will be lost when the Colab session ends — download them manually.')
        NALAPRO_DIR = '/content'
        os.chdir(NALAPRO_DIR)
    _IN_COLAB = True
    CKPT_BASE = '/content'
    print(f'Checkpoint base (fast local)   → {CKPT_BASE}')
    missing = [f for f in ['part1_results.csv', 'part2_results.csv',
                            'part3_results.csv', 'part4_results_all.csv']
               if not os.path.exists(f)]
    if missing:
        print(f'WARNING: missing prior-part CSVs: {missing}')
        print('  → Cross-part comparison will use placeholder values.')
except ModuleNotFoundError:
    CKPT_BASE = os.getcwd()
    print(f'Not in Colab. Working directory: {CKPT_BASE}')

---
## 1 · Model selection

We default to **`meta-llama/Llama-3.2-1B-Instruct`** — freely accessible with no HuggingFace
access request, and trains in ~10 minutes on a Colab T4.

To replicate Part 4's exact model (Llama-3-8B), set `MODEL_ID` to
`"meta-llama/Meta-Llama-3-8B-Instruct"` and add your HF token to Colab Secrets.

| Model | Parameters | VRAM (4-bit) | Train time on T4 | Token needed |
|-------|-----------|--------------|-----------------|--------------|
| `TinyLlama-1.1B-Chat-v1.0` | 1.1 B | ~1.5 GB | ~8 min | No |
| `Llama-3.2-1B-Instruct` | 1.2 B | ~2 GB | ~10 min | Yes (gated) |
| `Llama-3.2-3B-Instruct` | 3.2 B | ~4 GB | ~20 min | Yes (gated) |
| `Meta-Llama-3-8B-Instruct` | 8 B | ~6 GB | ~45 min | Yes (gated) |

In [ ]:
# ── Choose model ─────────────────────────────────────────────────────────────
# Option A: fully open, no token required (default)
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Option B: Llama-3.2-1B — requires HF account + accepting licence at
#   https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct
# MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"

# Option C: same model as Part 4, requires HF token + Llama access
# MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"

# ── HuggingFace token (only needed for Options B / C) ────────────────────────
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = (userdata.get('HF_TOKEN') or '').strip() or None
except Exception:
    HF_TOKEN = (os.environ.get('HF_TOKEN', '') or '').strip() or None

if HF_TOKEN:
    print(f"Token found: {HF_TOKEN[:4]}...{HF_TOKEN[-4:]} (length {len(HF_TOKEN)})")
    try:
        from huggingface_hub import login
        login(token=HF_TOKEN)
        print("HuggingFace login successful.")
    except Exception as e:
        print(f"HuggingFace login failed: {e}")
        print("Continuing without login — only open models (Option A) will work.")
        HF_TOKEN = None
else:
    print("No HF_TOKEN — using open model (no login required).")

print(f"\nModel selected: {MODEL_ID}")

---
## 2 · Data

In [ ]:
from sklearn.datasets import fetch_20newsgroups

REMOVE = ("headers", "footers", "quotes")
ng_train = fetch_20newsgroups(subset="train", remove=REMOVE, random_state=SEED)
ng_test  = fetch_20newsgroups(subset="test",  remove=REMOVE, random_state=SEED)

class_names = ng_train.target_names
NUM_CLASSES  = len(class_names)
CLASS_LIST   = "\n".join(f"  {c}" for c in class_names)

print(f"Train documents : {len(ng_train.data):,}")
print(f"Test documents  : {len(ng_test.data):,}")
print(f"Number of classes: {NUM_CLASSES}")

In [ ]:
# ── Stratified evaluation set — identical to Part 4 for direct comparison ────
# N_PER_CLASS=20 → 400 total docs, same seed, same sample as Part 4.
N_PER_CLASS = 20

by_class = defaultdict(list)
for text, label in zip(ng_test.data, ng_test.target):
    if text and len(text.strip()) > 10:
        by_class[label].append(text)

rng = random.Random(SEED)
eval_texts, eval_labels = [], []
for c in range(NUM_CLASSES):
    pool  = by_class[c]
    picks = rng.sample(pool, min(N_PER_CLASS, len(pool)))
    eval_texts.extend(picks)
    eval_labels.extend([c] * len(picks))

eval_labels = np.array(eval_labels)
print(f"Evaluation set: {len(eval_texts)} documents  ({N_PER_CLASS} per class)")

In [ ]:
# ── Filter training documents ─────────────────────────────────────────────────
# Remove empty or very short documents that result from header/footer stripping.
train_pairs = [
    (t, ng_train.target[i])
    for i, t in enumerate(ng_train.data)
    if t and len(t.strip()) >= 20
]
train_texts  = [p[0] for p in train_pairs]
train_labels = [p[1] for p in train_pairs]
print(f"Training examples after filtering: {len(train_texts):,}")

---
## 3 · QLoRA setup

**Code reference:** DataCamp — *Fine-Tuning Llama 3.1*
→ https://www.datacamp.com/tutorial/fine-tuning-llama-3-1
The 4-bit loading pattern, LoRA target modules, and SFTTrainer configuration below
follow that tutorial, adapted here for 20-class text classification instead of general chat fine-tuning.

### Step 1 — 4-bit quantisation with BitsAndBytes

| Config key | Value | Explanation |
|------------|-------|-------------|
| `load_in_4bit` | True | Store base model weights in 4-bit |
| `bnb_4bit_quant_type` | `"nf4"` | NormalFloat4 — optimal for normally distributed weights |
| `bnb_4bit_compute_dtype` | `bfloat16` | Upcasts to bf16 for actual forward/backward arithmetic |
| `bnb_4bit_use_double_quant` | True | Quantise the quantisation constants → saves ~0.4 bits/param |

### Step 2 — LoRA adapters (PEFT)

LoRA injects two small matrices **A** (r × d) and **B** (d × r) alongside each frozen linear
weight **W**. The effective weight becomes **W + (α/r)·BA** where:
- **r** (rank) = adapter capacity; `r=8` adds ~2 M params on a 1B model
- **α** (alpha) = scaling; usually set to 2 r so effective scale = 2

Only **A** and **B** receive gradients — the 4-bit base weights never change.

In [ ]:
# Adapted from: DataCamp — "Fine-Tuning Llama 3.1"
# https://www.datacamp.com/tutorial/fine-tuning-llama-3-1
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {MODEL_ID} in 4-bit quantisation…")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    token=HF_TOKEN,
)
base_model.config.use_cache = False  # disable KV-cache during training (saves memory)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=HF_TOKEN,
)
# Llama has no pad token by default — use EOS
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"   # right-padding during training

total_base = sum(p.numel() for p in base_model.parameters())
print(f"\nBase model loaded. Parameters: {total_base:,}")
print(f"GPU memory after loading: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# LoRA configuration follows the DataCamp tutorial pattern:
# https://www.datacamp.com/tutorial/fine-tuning-llama-3-1
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

# prepare_model_for_kbit_training:
#  • enables gradient checkpointing
#  • casts non-quantised layers (LayerNorm, embeddings) to float32 for numerical stability
model = prepare_model_for_kbit_training(base_model)

# Required when gradient_checkpointing=True with PEFT:
# PEFT freezes base weights, so the first trainable layer may have no grad_fn
# unless we explicitly tell PyTorch to propagate gradients into the input embeddings.
model.enable_input_require_grads()

LORA_R       = 8    # adapter rank
LORA_ALPHA   = 16   # scaling = alpha/r = 2 (standard practice)
LORA_DROPOUT = 0.05

# Target all projection layers in attention and MLP
TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",  # attention
    "gate_proj", "up_proj", "down_proj",       # MLP (Llama uses SwiGLU)
]

lora_config = LoraConfig(
    r             = LORA_R,
    lora_alpha    = LORA_ALPHA,
    lora_dropout  = LORA_DROPOUT,
    target_modules= TARGET_MODULES,
    bias          = "none",          # do not adapt bias vectors
    task_type     = TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)

# ── Parameter breakdown ───────────────────────────────────────────────────────
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
frozen    = total - trainable
print(f"Total parameters   : {total:>15,}")
print(f"Trainable (LoRA)   : {trainable:>15,}  ({100*trainable/total:.3f}%)")
print(f"Frozen (quantised) : {frozen:>15,}")
model.print_trainable_parameters()

In [ ]:
# ── LoRA parameter breakdown & GPU memory visualisation ──────────────────────
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel 1 — trainable LoRA vs frozen base model
categories  = ["Frozen base model\n(4-bit quantised)", "LoRA adapters\n(trainable, fp32)"]
param_counts = [frozen, trainable]
bar_colors   = ["#78909C", "#E91E63"]
bars = axes[0].bar(categories, param_counts, color=bar_colors, width=0.45)
axes[0].bar_label(bars, labels=[f"{v/1e6:.1f} M" for v in param_counts], padding=5, fontsize=11, fontweight="bold")
axes[0].set_title("Parameter Breakdown\nBase model frozen — only LoRA adapters update", fontsize=11, fontweight="bold")
axes[0].set_ylabel("Number of parameters")
axes[0].set_ylim(0, max(param_counts) * 1.18)
axes[0].grid(axis="y", alpha=0.3)
axes[0].tick_params(axis="x", labelsize=10)

# Annotate trainable %
axes[0].annotate(
    f"{100*trainable/total:.2f}% of total",
    xy=(1, trainable), xytext=(1, trainable * 1.08),
    ha="center", fontsize=10, color="#E91E63", fontweight="bold",
)

# Panel 2 — GPU VRAM usage
vram_used  = torch.cuda.memory_allocated() / 1e9
vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
vram_free  = vram_total - vram_used
vram_labels = ["Model + adapters\n(allocated)", "Free VRAM"]
vram_vals   = [vram_used, vram_free]
vram_colors = ["#E91E63", "#B0BEC5"]
bars2 = axes[1].bar(vram_labels, vram_vals, color=vram_colors, width=0.45)
axes[1].bar_label(bars2, fmt="%.1f GB", padding=5, fontsize=11, fontweight="bold")
axes[1].set_title(f"GPU Memory Usage\n(Total VRAM: {vram_total:.0f} GB)", fontsize=11, fontweight="bold")
axes[1].set_ylabel("VRAM (GB)")
axes[1].set_ylim(0, vram_total * 1.18)
axes[1].grid(axis="y", alpha=0.3)
axes[1].tick_params(axis="x", labelsize=10)

plt.suptitle("QLoRA Resource Footprint — Before Training", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("part5_qlora_breakdown.png", dpi=120, bbox_inches="tight")
plt.show()
print("→ Saved part5_qlora_breakdown.png  (include in the report)")

---
## 4 · Format training data as instruction-following examples

Each training document is wrapped in the Llama-3 Instruct chat template.
The model learns to generate the correct category name given the system message and post.

The system message is identical to Part 4 so the model encounters the same task framing.

In [ ]:
# ── System message — same framing as Part 4 ─────────────────────────────────
SYSTEM_MSG = (
    "You are a precise text classifier.\n"
    "Classify the Usenet newsgroup post into exactly ONE of the 20 categories listed below.\n"
    "Output ONLY the category name on a single line — nothing else, no explanation, no punctuation.\n"
    f"\nCategories:\n{CLASS_LIST}"
)

from datasets import Dataset

print("Formatting training examples with Llama-3 chat template…")
formatted = []
for text, label_idx in zip(train_texts, train_labels):
    messages = [
        {"role": "system",    "content": SYSTEM_MSG},
        {"role": "user",      "content": f'Post:\n"""\n{text[:1500]}\n"""'},
        {"role": "assistant", "content": class_names[label_idx]},
    ]
    # add_generation_prompt=False: we include the full assistant turn for training
    formatted_str = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    formatted.append({"text": formatted_str})

train_dataset = Dataset.from_list(formatted)
print(f"Training dataset: {len(train_dataset):,} examples")
print("\nFirst example (first 600 chars):")
print(train_dataset[0]["text"][:600])

---
## 5 · SFT Training

**Code reference:** DataCamp — *Fine-Tuning Llama 3.1*
→ https://www.datacamp.com/tutorial/fine-tuning-llama-3-1

### Key hyper-parameters

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| `num_train_epochs` | 1 | One pass over the data is typically sufficient for instruction fine-tuning |
| `per_device_train_batch_size` | 4 | Fits in T4 VRAM with 4-bit model |
| `gradient_accumulation_steps` | 4 | Effective batch size = 16 |
| `learning_rate` | 2e-4 | Higher than full fine-tune — LoRA adapters are randomly initialised |
| `optim` | `paged_adamw_8bit` | 8-bit paged AdamW halves optimizer memory vs. standard AdamW |
| `gradient_checkpointing` | True | Trades compute for VRAM (recomputes activations on backward pass) |
| `max_seq_length` | 512 | Truncates long posts to keep sequences uniform and save VRAM |

In [ ]:
# SFTTrainer setup adapted from DataCamp tutorial:
# https://www.datacamp.com/tutorial/fine-tuning-llama-3-1
# Key difference: we classify 20-Newsgroups categories instead of general chat fine-tuning.
#
# trl breaks its API frequently; we use inspect to check what each version accepts
# rather than relying on version strings.

import inspect
from trl import SFTTrainer

MAX_SEQ_LENGTH = 512

# Ensure sequences are truncated even if SFTConfig no longer accepts max_seq_length
tokenizer.model_max_length = MAX_SEQ_LENGTH

SHARED_ARGS = dict(
    output_dir                    = f"{CKPT_BASE}/ckpt_part5_qlora",
    num_train_epochs              = 1,
    per_device_train_batch_size   = 4,
    gradient_accumulation_steps   = 4,
    gradient_checkpointing        = True,
    gradient_checkpointing_kwargs = {"use_reentrant": False},
    optim                         = "paged_adamw_8bit",
    learning_rate                 = 2e-4,
    lr_scheduler_type             = "cosine",
    warmup_steps                  = 10,
    weight_decay                  = 0.001,
    fp16                          = not torch.cuda.is_bf16_supported(),
    bf16                          = torch.cuda.is_bf16_supported(),
    logging_steps                 = 25,
    save_strategy                 = "epoch",
    save_total_limit              = 1,
    report_to                     = "none",
    seed                          = SEED,
)

# Build config — inspect what SFTConfig accepts so we never pass unknown kwargs
try:
    from trl import SFTConfig
    _cfg_params = inspect.signature(SFTConfig).parameters
    _cfg_kwargs  = dict(**SHARED_ARGS)
    if "max_seq_length"     in _cfg_params: _cfg_kwargs["max_seq_length"]     = MAX_SEQ_LENGTH
    if "dataset_text_field" in _cfg_params: _cfg_kwargs["dataset_text_field"] = "text"
    train_config = SFTConfig(**_cfg_kwargs)
    _using = "SFTConfig"
except ImportError:
    from transformers import TrainingArguments
    train_config = TrainingArguments(**SHARED_ARGS)
    _using = "TrainingArguments"

# Build trainer — inspect what SFTTrainer.__init__ accepts
_tr_params  = inspect.signature(SFTTrainer.__init__).parameters
_tr_kwargs  = dict(model=model, args=train_config, train_dataset=train_dataset)
if   "processing_class" in _tr_params: _tr_kwargs["processing_class"] = tokenizer
elif "tokenizer"        in _tr_params: _tr_kwargs["tokenizer"]        = tokenizer
# Only pass these if SFTConfig didn't already absorb them
if "max_seq_length"     in _tr_params and "max_seq_length"     not in _cfg_kwargs: _tr_kwargs["max_seq_length"]     = MAX_SEQ_LENGTH
if "dataset_text_field" in _tr_params and "dataset_text_field" not in _cfg_kwargs: _tr_kwargs["dataset_text_field"] = "text"

trainer = SFTTrainer(**_tr_kwargs)
print(f"SFTTrainer ready ({_using})")

print(f"\nTraining on {len(train_dataset):,} examples  |  max_seq_length={MAX_SEQ_LENGTH}")
print(f"GPU memory before training: {torch.cuda.memory_allocated()/1e9:.2f} GB\n")
trainer.train()

In [ ]:
# ── Save the LoRA adapter ─────────────────────────────────────────────────────
ADAPTER_PATH = f"{CKPT_BASE}/qlora_adapter_part5"
trainer.model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"LoRA adapter saved to: {ADAPTER_PATH}")
print(f"GPU memory after training: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
import matplotlib.pyplot as plt

train_log = [e for e in trainer.state.log_history
             if "loss" in e and "eval_loss" not in e]
steps  = [e["step"] for e in train_log]
losses = [e["loss"]  for e in train_log]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(steps, losses, color="#E91E63", linewidth=1.5, label="Training loss")
ax.set_title("QLoRA Training Loss per Step", fontsize=13, fontweight="bold")
ax.set_xlabel("Step")
ax.set_ylabel("Cross-entropy loss")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig("part5_training_loss.png", dpi=120, bbox_inches="tight")
plt.show()
print("→ Saved part5_training_loss.png")

---
## 6 · Evaluation

We evaluate on the same **400-document stratified subset** used in Part 4
(20 docs × 20 classes, same seed), so results are directly comparable.

**Protocol:**
1. Format each post as an instruction prompt (system + user, no assistant answer)
2. Generate up to 15 new tokens with greedy decoding (temperature → 0)
3. Decode only the newly generated tokens
4. Parse with the same 3-tier label parser used in Part 4

In [ ]:
def parse_label(raw: str) -> int:
    """
    Robust label parser — identical to Part 4 for direct comparability.
    Returns class index (0–19) or -1 if no match found.
    """
    if not raw:
        return -1
    text = raw.strip().splitlines()[0].strip().rstrip(".,;:!?")

    # 1. Exact match (case-insensitive)
    for i, c in enumerate(class_names):
        if text.lower() == c.lower():
            return i

    # 2. Substring match
    text_lower = text.lower()
    for i, c in enumerate(class_names):
        if c.lower() in text_lower:
            return i

    # 3. Token-level match
    tokens = re.split(r"[\s/,;:]+", text_lower)
    for tok in tokens:
        for i, c in enumerate(class_names):
            if tok == c.lower():
                return i

    return -1

In [ ]:
import gc

# Re-enable KV-cache and set model to eval mode for faster inference
model.config.use_cache = True
model.eval()

# Left-padding for generation (ensures the prompt is correctly positioned)
tokenizer.padding_side = "left"

def predict_qlora(post: str, max_new_tokens: int = 15) -> str:
    """
    Generate a category label for a newsgroup post using the fine-tuned model.
    Returns the raw decoded string (before label parsing).
    """
    messages = [
        {"role": "system", "content": SYSTEM_MSG},
        {"role": "user",   "content": f'Post:\n"""\n{post[:2000]}\n"""'},
    ]
    # add_generation_prompt=True appends the assistant header — prompts the model to respond
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=512
    ).to(DEVICE)

    with torch.no_grad():
        # do_sample=False → greedy decoding; no temperature/top_p needed
        output_ids = model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            do_sample      = False,
            pad_token_id   = tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens (skip the prompt)
    new_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip()

# Smoke test
sample_post = eval_texts[0]
raw_test    = predict_qlora(sample_post)
print(f"True label : {class_names[eval_labels[0]]}")
print(f"Prediction : {repr(raw_test)}")
print(f"Parsed     : {class_names[parse_label(raw_test)] if parse_label(raw_test) >= 0 else 'UNPARSED'}")

In [ ]:
print(f"Running QLoRA inference on {len(eval_texts)} evaluation documents…")
print("(Expect ~1–3 seconds per document on a T4 GPU)")

raw_outputs = []
qlora_preds = []

for post in tqdm(eval_texts, desc="QLoRA inference"):
    try:
        raw = predict_qlora(post)
    except Exception as e:
        print(f"Inference error: {e}")
        raw = ""
    raw_outputs.append(raw)
    qlora_preds.append(parse_label(raw))

qlora_preds = np.array(qlora_preds)
unparseable = (qlora_preds == -1).sum()
print(f"\nUnparseable responses: {unparseable} / {len(qlora_preds)}  ({100*unparseable/len(qlora_preds):.1f}%)")

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

preds_safe = np.where(qlora_preds == -1, NUM_CLASSES, qlora_preds)

qlora_acc            = accuracy_score(eval_labels, preds_safe)
qlora_f1             = f1_score(eval_labels, preds_safe, average="macro",
                                labels=list(range(NUM_CLASSES)))
qlora_unparseable_pct = 100 * (qlora_preds == -1).sum() / len(qlora_preds)

print(f"QLoRA fine-tune → accuracy = {qlora_acc:.4f}   macro-F1 = {qlora_f1:.4f}   unparseable = {qlora_unparseable_pct:.1f}%")

In [ ]:
import pandas as pd

def _best_row(path, f1_col="Macro-F1"):
    try:
        df = pd.read_csv(path)
        col = f1_col if f1_col in df.columns else df.columns[-1]
        return df.loc[pd.to_numeric(df[col], errors="coerce").idxmax()]
    except Exception:
        return None

def _get(row, *keys):
    if row is None:
        return None
    for k in keys:
        if k in row.index:
            try:
                return float(row[k])
            except (ValueError, TypeError):
                return row[k]
    return None

p1 = _best_row("part1_results.csv", "Macro-F1")
p2 = _best_row("part2_results.csv", "Test Macro-F1")
p3 = _best_row("part3_results.csv", "Test F1")

# Load Part 4 results
try:
    p4 = pd.read_csv("part4_results_all.csv")
    p4_zs = p4[p4["Experiment"].str.contains("zero-shot", case=False)].iloc[0]
    p4_fs = p4[p4["Experiment"].str.contains("shot/class", case=False)].iloc[0]
    p4_zs_acc = float(p4_zs["Test Accuracy"]); p4_zs_f1 = float(p4_zs["Test Macro-F1"])
    p4_fs_acc = float(p4_fs["Test Accuracy"]); p4_fs_f1 = float(p4_fs["Test Macro-F1"])
    p4_zs_unp = float(p4_zs.get("Unparseable %", 0) or 0)
    p4_fs_unp = float(p4_fs.get("Unparseable %", 0) or 0)
    p4_zs_label = str(p4_zs["Experiment"])
    p4_fs_label = str(p4_fs["Experiment"])
except Exception:
    p4_zs_acc = p4_zs_f1 = p4_zs_unp = 0.0
    p4_fs_acc = p4_fs_f1 = p4_fs_unp = 0.0
    p4_zs_label = "Part 4 — Llama-3 zero-shot"
    p4_fs_label = "Part 4 — Llama-3 1-shot/class"

short_model = MODEL_ID.split("/")[-1]

RESULTS = [
    {"Experiment": f"Part 1 — {_get(p1,'Experiment') or 'TF-IDF'} + SimpleNN",
     "Test Accuracy": _get(p1,"Test Acc","Test Accuracy") or 0.0,
     "Test Macro-F1": _get(p1,"Macro-F1","Test Macro-F1") or 0.0,
     "Unparseable %": None, "Trainable Params": "N/A"},
    {"Experiment": f"Part 2 — {_get(p2,'Experiment') or 'BERT full fine-tune'}",
     "Test Accuracy": _get(p2,"Test Acc","Test Accuracy") or 0.0,
     "Test Macro-F1": _get(p2,"Test Macro-F1","Test F1") or 0.0,
     "Unparseable %": None, "Trainable Params": "N/A"},
    {"Experiment": "Part 3 — MLM-adapted BERT",
     "Test Accuracy": _get(p3,"Test Acc","Test Accuracy") or 0.0,
     "Test Macro-F1": _get(p3,"Test F1","Test Macro-F1") or 0.0,
     "Unparseable %": None, "Trainable Params": "N/A"},
    {"Experiment": p4_zs_label,
     "Test Accuracy": p4_zs_acc, "Test Macro-F1": p4_zs_f1,
     "Unparseable %": round(p4_zs_unp, 1), "Trainable Params": 0},
    {"Experiment": p4_fs_label,
     "Test Accuracy": p4_fs_acc, "Test Macro-F1": p4_fs_f1,
     "Unparseable %": round(p4_fs_unp, 1), "Trainable Params": 0},
    {"Experiment": f"Part 5 — {short_model} QLoRA fine-tune",
     "Test Accuracy": qlora_acc, "Test Macro-F1": qlora_f1,
     "Unparseable %": round(qlora_unparseable_pct, 1), "Trainable Params": trainable},
]

summary = pd.DataFrame(RESULTS)
print("\n── Cross-part results summary ──────────────────────────────────────────────")
print(summary.to_string(index=False))
summary.to_csv("part5_results_all.csv", index=False)
print("\n→ Saved part5_results_all.csv")

In [ ]:
# ── Prompting vs. fine-tuning — direct comparison on the same eval set ────────
import matplotlib.pyplot as plt
import numpy as np

methods      = ["Zero-shot\n(Part 4)", "Few-shot k=1\n(Part 4)", "QLoRA fine-tune\n(Part 5)"]
method_accs  = [p4_zs_acc, p4_fs_acc, qlora_acc]
method_f1s   = [p4_zs_f1,  p4_fs_f1,  qlora_f1]
method_colors= ["#FF9800", "#9C27B0", "#E91E63"]

x = np.arange(3)
width = 0.33

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, vals, ylabel, title in [
    (axes[0], method_accs, "Accuracy",  "Test Accuracy"),
    (axes[1], method_f1s,  "Macro-F1",  "Test Macro-F1"),
]:
    bars = ax.bar(x, vals, width=0.5, color=method_colors)
    ax.bar_label(bars, fmt="%.3f", padding=4, fontsize=11, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(methods, fontsize=11)
    ax.set_ylim(0, 1.12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.grid(axis="y", alpha=0.3)
    # Delta annotation between few-shot and QLoRA
    delta = vals[2] - vals[1]
    ax.annotate(
        f"+{delta:.3f}" if delta > 0 else f"{delta:.3f}",
        xy=(2, vals[2] + 0.04), ha="center", fontsize=11,
        color="#4CAF50" if delta > 0 else "#d32f2f", fontweight="bold",
    )

plt.suptitle(
    "Prompting vs. Fine-tuning — Same Llama-3 model, same 400-doc eval set\n"
    "(+Δ = improvement over few-shot)",
    fontsize=12, fontweight="bold"
)
plt.tight_layout()
plt.savefig("part5_prompting_vs_finetuning.png", dpi=120, bbox_inches="tight")
plt.show()
print("→ Saved part5_prompting_vs_finetuning.png  (include in the report)")

In [ ]:
import matplotlib.pyplot as plt

labels = [r["Experiment"].replace(" — ", "\n") for r in RESULTS]
accs   = [r["Test Accuracy"] for r in RESULTS]
f1s    = [r["Test Macro-F1"] for r in RESULTS]
colors = ["#FF6B35", "#2196F3", "#4CAF50", "#FF9800", "#9C27B0", "#E91E63"]

x = range(len(RESULTS))
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

for ax, vals, title in [(axes[0], accs, "Test Accuracy"), (axes[1], f1s, "Test Macro-F1")]:
    bars = ax.bar(x, vals, color=colors, width=0.55)
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=8.5, rotation=15, ha="right")
    ax.set_ylim(0, 1.08)
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_ylabel(title)
    ax.grid(axis="y", alpha=0.3)

plt.suptitle(
    "Performance across all project parts (incl. QLoRA bonus)\n(20-Newsgroups test set)",
    fontsize=13
)
plt.tight_layout()
plt.savefig("part5_all_results.png", dpi=120, bbox_inches="tight")
plt.show()
print("→ Saved part5_all_results.png")

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

mask = qlora_preds != -1
cm = confusion_matrix(eval_labels[mask], qlora_preds[mask],
                      labels=list(range(NUM_CLASSES)), normalize="true")

fig, ax = plt.subplots(figsize=(13, 11))
disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
disp.plot(ax=ax, xticks_rotation=90, cmap="PuRd", colorbar=False, values_format=".2f")
ax.set_title(f"Llama-3 QLoRA fine-tune — normalised confusion matrix",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("part5_confusion_matrix.png", dpi=120, bbox_inches="tight")
plt.show()
print("→ Saved part5_confusion_matrix.png")

In [ ]:
from sklearn.metrics import classification_report
import pandas as pd
import matplotlib.pyplot as plt

mask = qlora_preds != -1
report = classification_report(eval_labels[mask], qlora_preds[mask],
                               target_names=class_names, output_dict=True)
per_class_df = pd.DataFrame({
    "class":    class_names,
    "f1_score": [report[c]["f1-score"] for c in class_names],
    "support":  [report[c]["support"]  for c in class_names],
}).sort_values("f1_score")

print(per_class_df.to_string(index=False))
print(f"\nHardest classes: {per_class_df['class'].head(3).tolist()}")
print(f"Easiest classes: {per_class_df['class'].tail(3).tolist()}")

fig, ax = plt.subplots(figsize=(10, 6))
bar_colors = ["#d32f2f" if f < 0.6 else "#FF9800" if f < 0.75 else "#4CAF50"
              for f in per_class_df["f1_score"]]
ax.barh(per_class_df["class"], per_class_df["f1_score"], color=bar_colors)
ax.set_xlabel("F1 score", fontsize=12)
ax.set_title("Per-class F1 — Llama-3 QLoRA fine-tune", fontsize=12, fontweight="bold")
ax.axvline(x=per_class_df["f1_score"].mean(), color="navy", linestyle="--", label="Macro-F1")
ax.legend()
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("part5_per_class_f1.png", dpi=120, bbox_inches="tight")
plt.show()
print("→ Saved part5_per_class_f1.png")

In [ ]:
# ── Qualitative inspection — correct, wrong, and confusable predictions ────────
print("=" * 82)
print("QUALITATIVE EXAMPLES FOR THE REPORT")
print("=" * 82)

correct_shown = wrong_shown = 0
for i in range(len(eval_texts)):
    if correct_shown >= 3 and wrong_shown >= 3:
        break
    pred_idx  = qlora_preds[i]
    true_idx  = int(eval_labels[i])
    pred_name = class_names[pred_idx] if pred_idx >= 0 else "UNPARSED"
    true_name = class_names[true_idx]
    is_correct = (pred_idx == true_idx)

    if is_correct and correct_shown < 3:
        tag = "✓ CORRECT"
        correct_shown += 1
    elif not is_correct and wrong_shown < 3:
        tag = "✗ WRONG  "
        wrong_shown += 1
    else:
        continue

    print(f"\n[{tag}]")
    print(f"  True label  : {true_name}")
    print(f"  Prediction  : {pred_name}")
    print(f"  Raw output  : {repr(raw_outputs[i][:80])}")
    print(f"  Post excerpt: {eval_texts[i][:250].replace(chr(10), ' ')}")

print()
print("(These examples can be used directly in the report discussion section)")

---
## ⚠️ Teardown — Run this before closing the notebook

> ### 🔴 STOP AND READ BEFORE YOU CLOSE
>
> **Google Colab charges per GPU-hour while the runtime is connected.**
> Even if you're not running cells, the runtime keeps billing.
>
> | Time forgotten | Approx cost (Colab Pro, T4 GPU) |
> |---|---|
> | 1 hour | ~$0.35 |
> | 1 day | ~$8 |
> | 1 week | ~$56 |
>
> **Run the two cells below, then disconnect your runtime.**

### After running the teardown cells
Go to **Runtime → Disconnect and delete runtime** in the Colab menu bar.

In [ ]:
# ── Step 1: Free GPU VRAM ─────────────────────────────────────────────────────
import gc, torch

print("Freeing Llama-3 QLoRA model from GPU VRAM…")

for _varname in ["model", "base_model", "trainer"]:
    if _varname in globals():
        obj = globals()[_varname]
        if hasattr(obj, "model") and hasattr(obj.model, "cpu"):
            try: obj.model.cpu()
            except: pass
        elif hasattr(obj, "cpu"):
            try: obj.cpu()
            except: pass
        del globals()[_varname]
        print(f"  ✓ Deleted {_varname}")

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    allocated = torch.cuda.memory_allocated() / 1e6
    print(f"CUDA cache cleared. GPU memory still allocated: {allocated:.1f} MB")

print("✓ Step 1 complete.")

In [ ]:
# ── Step 2: Delete checkpoint directories ─────────────────────────────────────
import shutil, os

ckpt_dirs = [
    f"{CKPT_BASE}/ckpt_part5_qlora",
    ADAPTER_PATH,
]

total_freed = 0
for d in ckpt_dirs:
    if os.path.isdir(d):
        size_mb = sum(
            os.path.getsize(os.path.join(dp, f))
            for dp, _, files in os.walk(d) for f in files
        ) / 1e6
        shutil.rmtree(d)
        total_freed += size_mb
        print(f"✓ Deleted {d}  (freed ~{size_mb:.0f} MB)")
    else:
        print(f"  {d} not found (already deleted — OK)")

print(f"\nTotal freed: ~{total_freed:.0f} MB")
print()
print("=" * 65)
print("PART 5 TEARDOWN CHECKLIST")
print("=" * 65)
for item in [
    "Llama-3 QLoRA model deleted from GPU VRAM",
    "Trainer object deleted",
    "LoRA adapter checkpoint directory deleted",
    "bitsandbytes 4-bit model freed via gc.collect()",
]:
    print(f"  ✓ {item}")
print()
print("FINAL STEP (manual):")
print("  → Colab menu: Runtime → Disconnect and delete runtime")
print("  → Verify no unexpected spend: https://console.cloud.google.com/billing")
print()
print("Part 5 teardown complete.")